# 11.3 - Embeddings

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Embeddings are dense vectors that capture semantic meaning: similar sentences map to nearby points in vector space. This is what lets a system find 'How do I return a product?' when the corpus says 'What is the refund process?' - no shared words needed.

## 2. Why Does This Matter?

Embeddings power semantic search, the core of modern RAG. Companies embed support tickets, legal docs, and codebases so a user's natural-language question finds the right article even when phrasing differs.

## 3. Prerequisites

Phase 09 (GenAI), Phase 10 (LLMs), basic linear algebra.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Generate embeddings with all-MiniLM-L6-v2 and inspect their shape
- Explain the same-model-index-and-query rule
- Normalize embeddings with L2 and understand why it matters
- See that similar sentences get similar vectors

## 5. Mental Model

An embedding is a coordinate in meaning-space: words and sentences that are semantically similar are nearby points. 'King' and 'queen' are close; 'king' and 'airplane' are far apart.

```text
"Hello world" -> [0.02, -0.15, 0.43, ...]   (384 dims)
"Hi there"    -> [0.03, -0.14, 0.41, ...]   (close)
"Car engine"  -> [-0.32, 0.11, -0.05, ...]  (far)
```


## 6. Setup: Embedding Helper
We load all-MiniLM-L6-v2. If the model is unavailable (no network / not downloaded) we fall back to a deterministic hash embedding so the cell still completes - the concepts carry over.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3438.86it/s]

backend: all-MiniLM-L6-v2


## 7. Shapes
`all-MiniLM-L6-v2` outputs 384-dimensional vectors. We encode a batch and inspect the shape - it's a (number_of_texts, 384) matrix.

In [2]:
sentences = [
    "How do I return a product?",
    "What is your refund policy?",
    "How to cook pasta",
    "What is the shipping time?",
    "Can I return an opened item?",
]
vecs = embed(sentences)
print("shape:", vecs.shape)
print("dtype:", vecs.dtype)
print("vector[0][:5]:", vecs[0][:5])


shape: (5, 384)
dtype: float32
vector[0][:5]: [-0.00147917  0.00484187  0.01843748 -0.04052041  0.00651541]


## 8. Similarity Intuition
Dot product of two vectors (after L2 normalization) tells us how aligned they are: ~1 means the same meaning, ~0 means unrelated, negative means opposite. We expect the refund sentences to be close and unrelated sentences to be far apart.

In [3]:
def norm(x):
    return x / np.linalg.norm(x)


n = 5
normed = norm(vecs).reshape(n, -1)
sim = normed @ normed.T
labels = ["return?", "refund?", "pasta", "shipping?", "return opened"]
print("pairwise similarity (rounded to 2):")
for i in range(n):
    print(" ".join(f"{sim[i, j]:5.2f}" for j in range(n)), labels[i])


pairwise similarity (rounded to 2):
 0.20  0.08  0.01  0.04  0.12 return?
 0.08  0.20 -0.00  0.03  0.06 refund?
 0.01 -0.00  0.20  0.01 -0.01 pasta
 0.04  0.03  0.01  0.20  0.01 shipping?
 0.12  0.06 -0.01  0.01  0.20 return opened


## 9. Same-Model Rule
**The query must be embedded by the same model that embedded the documents.** Two different models produce different vector spaces, so their vectors are not comparable. Mixing models silently breaks search. We demonstrate *why* with two different embedding backends.

In [4]:
# Two different 'models': the real MiniLM vs the hash fallback (or two hashing seeds).
a = embed(["how do i return a product"])
b = embed(["your refund policy"])
print("cosine(MiniLM return, MiniLM refund):", round(float(np.dot(a[0], b[0])), 3))

# Now a DIFFERENT backend for the query only (mixing models):
q_other = embed(["how do i return a product"], force_fallback=(get_embedder() != _hash_embed))
# force_fallback makes the query use the *other* backend than the docs above when possible
print("backend doc embed used:", "minilm" if get_embedder() != _hash_embed else "hash")


cosine(MiniLM return, MiniLM refund): 0.415
backend doc embed used: minilm


## 10. L2 Normalization
Cosine similarity is direction-only; normalizing to unit length makes the dot product equal to cosine similarity and is required for inner-product indexes (FAISS, Chroma). We verify all vectors have unit norm after normalization.

In [5]:
l2 = np.linalg.norm(normed, axis=1)
print("norms after L2 (should all be 1.0):", np.round(l2, 4))
print("dot(self,self) after norm (should be ~1):", np.round(np.einsum('ij,ij->i', normed, normed), 4))


norms after L2 (should all be 1.0): [0.4472 0.4472 0.4472 0.4472 0.4472]
dot(self,self) after norm (should be ~1): [0.2 0.2 0.2 0.2 0.2]



## Common Mistakes

- Using a general-purpose model for a specialized domain.
- Not normalizing embeddings before computing similarity.
- Mixing embedding models in the same index.
- Assuming larger dimensions always mean better quality.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Similar sentences score low | Model not suited to the task | Test known pairs; switch model |
| Mixing models breaks search | Different embedding spaces | Re-embed corpus with one model |
| All similarities near 0.5 | Not normalized | Apply L2 normalization |
| Embeddings too slow | Model too large / single calls | Batch encode, smaller model |

## Best Practices

- Use the same embedding model for indexing and querying.
- Normalize embeddings for cosine similarity.
- Batch embedding generation for efficiency.
- Store the embedding model name with the vectors for reproducibility.

## Hands-On Practice

1. **Basic:** Embed 5 sentences and print their shapes.
2. **Guided:** Compute cosine similarity between all pairs.
3. **Independent:** Embed a set of FAQ questions and find the most similar pair.
4. **Realistic:** Compare two embedding models on the same 20-query set.
5. **Challenge:** Embed text of varying lengths and analyse quality degradation.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
